# 48-hour sessioned Doric simulation

This example creates a synthetic Doric file for a 48-hour schedule with 10 minutes of active recording followed by a 20 minute gap. The generated file contains one Doric `SeriesNNNN` group for each active recording session. The final 20 minute dark interval is part of the 48-hour schedule, but it is not stored as sampled data.

The calcium signal includes an additive tonic component with amplitude `1.0` and a 12-hour period. Digital IO channel 1 contains two behavior event types: `button` is encoded by one TTL pulse, and `treat taken` is encoded by two TTL pulses. Pulses are 50 ms wide, with a 50 ms off interval separating pulses in a multi-pulse sequence.

In [ ]:
from pathlib import Path

import numpy as np

from circadian_fiber_photometry import load_doric
from circadian_fiber_photometry.simulation import (
    SyntheticDoricConfig,
    SyntheticSignalConfig,
    SyntheticTTLBehaviorCodeConfig,
    SyntheticTTLBehaviorEventConfig,
    add_gaussian_noise,
    add_tonic_component,
    generate_synthetic_doric,
)


In [ ]:
total_schedule_hours = 48
session_duration_seconds = 10 * 60
inter_series_gap_seconds = 20 * 60
cycle_seconds = session_duration_seconds + inter_series_gap_seconds
series_count = int(total_schedule_hours * 60 * 60 / cycle_seconds)

fs = 60.0
channel_count = 1
tonic_period_hours = 12
tonic_frequency_hz = 1 / (tonic_period_hours * 60 * 60)

isosbestic_noise_std = 0.005
calcium_noise_std = 0.020
analog_in_noise_std = 0.050

ttl_pulse_width_seconds = 0.050
ttl_pulse_off_interval_seconds = 0.050
ttl_pulse_width_samples = int(round(ttl_pulse_width_seconds * fs))
ttl_pulse_off_samples = int(round(ttl_pulse_off_interval_seconds * fs))

active_recording_hours = series_count * session_duration_seconds / 3600
schedule_hours_including_final_gap = series_count * cycle_seconds / 3600

assert series_count == 96
assert np.isclose(active_recording_hours, 16.0)
assert np.isclose(schedule_hours_including_final_gap, total_schedule_hours)
assert ttl_pulse_width_samples == 3
assert ttl_pulse_off_samples == 3


In [ ]:
signal = SyntheticSignalConfig(
    bleaching_fraction=0.0,
    artifact_amplitude=0.0,
    circadian_amplitude=0.0,
    noise_std=0.0,
    analog_noise_std=0.0,
    transient_rate_per_minute=0.0,
    transient_amplitude=0.0,
)
signal = add_tonic_component(
    signal,
    amplitude=1.0,
    frequency_hz=tonic_frequency_hz,
    name='12-hour tonic component',
)
signal = add_gaussian_noise(
    signal,
    isosbestic_std=isosbestic_noise_std,
    calcium_std=calcium_noise_std,
    analog_in_std=analog_in_noise_std,
    name='measurement noise',
)

config = SyntheticDoricConfig(
    series_count=series_count,
    session_duration_seconds=session_duration_seconds,
    inter_series_gap_seconds=inter_series_gap_seconds,
    fs=fs,
    channel_count=channel_count,
    seed=123,
    signal=signal,
    ttl_behavior_codes=(
        SyntheticTTLBehaviorCodeConfig('button', channel=1, pulse_count=1),
        SyntheticTTLBehaviorCodeConfig('treat taken', channel=1, pulse_count=2),
    ),
    ttl_behavior_events=(
        SyntheticTTLBehaviorEventConfig('button', start_seconds=(60.0, 300.0)),
        SyntheticTTLBehaviorEventConfig('treat taken', start_seconds=(180.0, 420.0)),
    ),
    ttl_pulse_width_seconds=ttl_pulse_width_seconds,
    ttl_pulse_off_interval_seconds=ttl_pulse_off_interval_seconds,
)

cwd = Path.cwd()
output_dir = cwd / 'generated' if cwd.name == 'examples' else cwd / 'examples' / 'generated'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / '48_hour_tonic_session_simulation.doric'

summary = generate_synthetic_doric(output_path, config, overwrite=True)


In [ ]:
dataset = load_doric(output_path)
session_start_hours = dataset.session_start_times / 3600
calcium_session_means = dataset.calcium_465[:, 0, :].mean(axis=0)

button_events = [
    event for event in summary.ttl_behavior_events if event.code_name == 'button'
]
treat_events = [
    event for event in summary.ttl_behavior_events if event.code_name == 'treat taken'
]
first_treat_event = treat_events[0]
inter_pulse_gap_seconds = (
    first_treat_event.pulse_sample_indices[1]
    - (first_treat_event.pulse_sample_indices[0] + ttl_pulse_width_samples)
) / fs

np.testing.assert_allclose(summary.session_start_times[0], 0.0)
np.testing.assert_allclose(np.diff(summary.session_start_times), cycle_seconds)
np.testing.assert_allclose(dataset.session_start_times, summary.session_start_times)
np.testing.assert_allclose(schedule_hours_including_final_gap, total_schedule_hours)
np.testing.assert_allclose(active_recording_hours, 16.0)
np.testing.assert_allclose(dataset.fs, fs)
np.testing.assert_allclose(inter_pulse_gap_seconds, ttl_pulse_off_interval_seconds)
assert signal.gaussian_noise[0].calcium_std == calcium_noise_std
assert signal.gaussian_noise[0].isosbestic_std == isosbestic_noise_std
assert len(button_events) == series_count * 2
assert len(treat_events) == series_count * 2
assert button_events[0].pulse_sample_indices.size == 1
assert first_treat_event.pulse_sample_indices.size == 2

print(f'Generated file: {output_path}')
print(f'Sessions: {summary.series_count}')
print(f'Sampling frequency: {dataset.fs:.1f} Hz')
print(f'Session duration: {session_duration_seconds / 60:.1f} minutes')
print(f'Gap duration: {inter_series_gap_seconds / 60:.1f} minutes')
print(f'Session start spacing: {np.median(np.diff(summary.session_start_times)) / 60:.1f} minutes')
print(f'Active sampled recording time: {active_recording_hours:.1f} hours')
print(f'Schedule including final dark interval: {schedule_hours_including_final_gap:.1f} hours')
print(f'Tonic period: {tonic_period_hours:.1f} hours')
print(f'Tonic frequency: {tonic_frequency_hz:.8g} Hz')
print(f'Gaussian noise std, 405 channel: {isosbestic_noise_std:g}')
print(f'Gaussian noise std, 465 channel: {calcium_noise_std:g}')
print(f'Gaussian noise std, analog input: {analog_in_noise_std:g}')
print('TTL event codes: button = 1 pulse; treat taken = 2 pulses')
print(f'TTL pulse width: {ttl_pulse_width_seconds * 1000:.0f} ms')
print(f'TTL off interval between pulses: {ttl_pulse_off_interval_seconds * 1000:.0f} ms')
print(f'Button events: {len(button_events)} total; treat taken events: {len(treat_events)} total')

calcium_session_means[:10]


## Raw signal view

The loaded Doric dataset stores traces as `(samples, channels, sessions)`. This cell extracts the first channel's raw isosbestic and experimental traces, concatenates sessions along absolute experiment time, and plots both with Plotly Resampler so the full recording can be explored without manually downsampling the data.

In [ ]:
import socket

import plotly.graph_objects as go
from plotly_resampler import FigureResampler


def free_local_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(('127.0.0.1', 0))
        return int(sock.getsockname()[1])


raw_time_hours = dataset.timestamps.T.reshape(-1) / 3600
raw_isosbestic_405 = dataset.isosbestic_405[:, 0, :].T.reshape(-1)
raw_experimental_465 = dataset.calcium_465[:, 0, :].T.reshape(-1)

expected_samples = summary.samples_per_series * summary.series_count
assert raw_time_hours.size == expected_samples
assert raw_isosbestic_405.size == expected_samples
assert raw_experimental_465.size == expected_samples

raw_signal_figure = FigureResampler(go.Figure())
raw_signal_figure.add_trace(
    go.Scattergl(
        name='405 nm isosbestic',
        mode='lines',
        line={'color': '#4575b4', 'width': 1},
    ),
    hf_x=raw_time_hours,
    hf_y=raw_isosbestic_405,
)
raw_signal_figure.add_trace(
    go.Scattergl(
        name='465 nm experimental',
        mode='lines',
        line={'color': '#d73027', 'width': 1},
    ),
    hf_x=raw_time_hours,
    hf_y=raw_experimental_465,
)
raw_signal_figure.update_layout(
    template='plotly_white',
    title='Raw simulated photometry signals',
    xaxis_title='Elapsed experiment time (hours)',
    yaxis_title='Raw signal (V)',
    legend_title_text='Signal',
    hovermode='x unified',
)
raw_signal_figure.update_xaxes(rangeslider_visible=True)

plot_config = {'scrollZoom': True, 'responsive': True}

if 'get_ipython' in globals():
    try:
        raw_signal_figure.show_dash(
            mode='inline',
            port=free_local_port(),
            config=plot_config,
        )
    except Exception as exc:
        print(f'Plotly Resampler inline display failed: {exc}')
        raw_signal_figure.show(renderer='notebook', config=plot_config)
